In [1]:
#!pip install pandas numpy missingno openpyxl unidecode

In [2]:
import os
import glob
import pandas as pd
import numpy as np
import re
import missingno as msno

In [3]:
import warnings
# Remove excel styles warnings that can cause issues with openpyxl when reading files with complex formatting
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


# 1. Vital signs #

### 2022 to 2023

In [4]:
import os
import glob
import pandas as pd
import numpy as np
import re

# -------CONSTANTS---------
DATA_PATH = "../Datanad/subset_data"

YEAR_MIN = 2022
YEAR_MAX = 2023

OUTPUT_PATH = "Datasets"
OUTPUT_FILE = f"df_param_vit_{YEAR_MIN}_{YEAR_MAX}.csv"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Step 1: File screening ---
all_files = glob.glob(os.path.join(DATA_PATH, "*.xlsx"))

files = [
    f for f in all_files
    if "CGJ" in os.path.basename(f).upper()
       and "052C" in os.path.basename(f).upper()
       and not os.path.basename(f).startswith("~$")
]
files = sorted(files)
print(f"--- CHECK : {len(files)} files found ---")

# --- Step 2: Load ---
dfs = []
for file in files:
    try:
        df = pd.read_excel(file, header=3)
        df.columns = df.columns.astype(str).str.strip()
        df['source_file'] = os.path.basename(file)
        dfs.append(df)
        print(f" ✅ loaded : {os.path.basename(file)}")
    except Exception as e:
        print(f" ❌ Error on {os.path.basename(file)} : {e}")

if not dfs:
    print("⚠️ Aucun fichier chargé — arrêt.")
    exit()

df_concat = pd.concat(dfs, ignore_index=True)
print(f"Total rows loaded: {len(df_concat)}")

# --- Step 3: Cleaning functions ---
def clean_numeric(x):
    if pd.isna(x) or str(x).lower() in ["na", "n/a", "", "nan", "none"]:
        return np.nan
    s = str(x).strip().lower().replace(',', '.')
    s = re.sub(r'[^0-9.]', '', s)
    try:
        return float(s) if s != "" else np.nan
    except:
        return np.nan

def merge_columns_regex(df, regex_candidates, new_col_name):
    series = None
    for pattern in regex_candidates:
        matching_cols = [col for col in df.columns if re.search(pattern, str(col), re.IGNORECASE)]
        for col in matching_cols:
            if series is None:
                series = df[col]
            else:
                series = series.combine_first(df[col])
    if series is not None:
        df[new_col_name] = series
    return df

# --- Step 4: Fusion ---
df_merged = df_concat.copy()

# 4a. Colonnes par position
df_merged["nda"]              = df_merged.iloc[:, 2]
df_merged["date_adm_vitals"]  = df_merged.iloc[:, 3]


# 4b. Colonnes par regex
columns_mapping = {
    "urine_dipstick":  [r"Bandelette\s*Urinaire"],
    "sbp":             [r"TA\s*Max\s*Bras", r"TA\s*Syst"],
    "dbp":             [r"TA\s*Min\s*Bras", r"TA\s*Diast"],
    "hr":              [r"Fr[ée]quence\s*Cardiaque"],
    "temp":            [r"Temp[ée]rature"],
    "sat":             [r"Saturation\s*P[ée]riph[ée]rique"],
    "rr":              [r"Fr[ée]quence\s*Respiratoire"],
    "o2_flow":         [r"D[ée]bit.*Oxyg[èe]ne"],
    "cap_blood_sugar": [r"Glyc[ée]mie\s*Digitale"],
    "hemocue":         [r"Hemocue"],
    "gcs":             [r"Glasgow", r"Score\s*de\s*Glasgow"],
    "pain":            [r"EN\."],
    "breathalyzer":    [r"Ethylotest"],
    "pupil_right":     [r"Pupillaire\s*Droit"],
    "pupil_left":      [r"Pupillaire\s*Gauche"],
}

for new_col, regex_list in columns_mapping.items():
    df_merged = merge_columns_regex(df_merged, regex_list, new_col)

# --- Step 5: Final columns ---
final_cols = ["nda", "date_adm_vitals"] + \
             list(columns_mapping.keys()) + ["source_file"]
df_final = df_merged[[col for col in final_cols if col in df_merged.columns]].copy()

# --- Step 6: Cleaning ---

# 1. Dates
df_final["date_adm_vitals"]  = pd.to_datetime(df_final["date_adm_vitals"],  errors="coerce")


# 2. Diagnostic années
print("=== Distribution des années dans date_adm_vitals ===")
print(df_final["date_adm_vitals"].dt.year.value_counts().sort_index())
print(f"\ndate_adm_vitals NaT : {df_final['date_adm_vitals'].isna().sum()}")


# 3. Year filter
df_final = df_final[
    df_final["date_adm_vitals"].dt.year.between(YEAR_MIN, YEAR_MAX)
]
print(f"Rows after year filter ({YEAR_MIN}-{YEAR_MAX}): {len(df_final)}")

# 4. Numerical columns
quant_columns = ["sbp", "dbp", "hr", "temp", "sat", "rr", "cap_blood_sugar",
                 "hemocue", "gcs", "pain", "breathalyzer", "pupil_right", "pupil_left", "o2_flow"]

for col in quant_columns:
    if col in df_final.columns:
        df_final[col] = df_final[col].apply(clean_numeric)

# 5. Urine dipstick
if "urine_dipstick" in df_final.columns:
    df_final["urine_dipstick"] = df_final["urine_dipstick"].astype(str).replace(
        ['nan', 'None', 'NaN', ''], np.nan
    )

# 6. Hospital
df_final["hospital"] = np.where(
    df_final["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL"
)

# --- Step 7: Sort and export ---
df_final = df_final.sort_values(by=["nda", "date_adm_vitals"]).reset_index(drop=True)
df_final.to_csv(os.path.join(OUTPUT_PATH, OUTPUT_FILE), index=False)

print(f"\n🚀 Done! {len(df_final)} rows.")
print(f"Columns: {df_final.columns.tolist()}")

# --- Checks ---
print("\n=== Vitals coverage ===")
quant_cols = [c for c in quant_columns if c in df_final.columns]
print(df_final[quant_cols].notna().sum().sort_values(ascending=False))

print("\n=== Urine dipstick check ===")
if "urine_dipstick" in df_final.columns:
    valides = df_final["urine_dipstick"].dropna()
    print(f"Filled rows: {len(valides)}")
    print(valides.value_counts().head(5))

print("\n=== Pupils check ===")
if all(c in df_final.columns for c in ["pupil_right", "pupil_left"]):
    print(df_final[["pupil_right", "pupil_left"]].describe())

--- CHECK : 2 files found ---


 ✅ loaded : CGJ 052c - tous Param Vitaux heure_adm_2022(2).xlsx


 ✅ loaded : CGJ 052c - tous Param Vitaux heure_adm_2023.xlsx
Total rows loaded: 59727
=== Distribution des années dans date_adm_vitals ===
date_adm_vitals
2022    30581
2023    29146
Name: count, dtype: int64

date_adm_vitals NaT : 0
Rows after year filter (2022-2025): 59727



🚀 Done! 59727 rows.
Columns: ['nda', 'date_adm_vitals', 'urine_dipstick', 'sbp', 'dbp', 'hr', 'temp', 'sat', 'rr', 'o2_flow', 'cap_blood_sugar', 'hemocue', 'gcs', 'pain', 'breathalyzer', 'pupil_right', 'pupil_left', 'source_file', 'hospital']

=== Vitals coverage ===
sbp                58333
dbp                58329
hr                 58180
sat                57876
temp               56782
gcs                53210
pain               49254
o2_flow            48179
cap_blood_sugar    21446
rr                 13190
pupil_right        12269
pupil_left         12240
hemocue             2109
breathalyzer        1723
dtype: int64

=== Urine dipstick check ===
Filled rows: 5314
urine_dipstick
Positive Sang          1280
Négative               1174
ECBU                    907
Positive Leucocytes     605
Beta HCG                461
Name: count, dtype: int64

=== Pupils check ===
        pupil_right    pupil_left
count  12269.000000  12240.000000
mean       2.576901      2.740523
std        0.89

# 2. Triage nurse files #

### from 2022 to 2023

In [7]:
#===================================================
# IOA TRIAGE FILES - HARMONIZED PROCESSING (2022-2025)
# New format: long → wide (no skiprows, named columns)
#===================================================
import os
import glob
import re
import pandas as pd
import numpy as np
import unidecode

# ----------------------------
# CONSTANTS
# ----------------------------
DATA_PATH = "../Datanad/subset_data"
YEAR_MIN = 2022
YEAR_MAX = 2023
os.makedirs("Datasets", exist_ok=True)
output_file = f"Datasets/df_ioa_{YEAR_MIN}_{YEAR_MAX}.csv"


# ----------------------------
# 1️⃣ FILE INVENTORY
# ----------------------------
def get_ioa_files(pattern="CGJ 030*"):
    files = glob.glob(os.path.join(DATA_PATH, pattern))
    files = [f for f in files if not os.path.basename(f).startswith("~$")]
    print(f"--- INVENTORY: {len(files)} files detected ---")
    return sorted(files)

# ----------------------------
# 2️⃣ LOAD AND CONCATENATE
# ----------------------------
def load_files(files):
    dfs = []
    for f in files:
        try:
            df_temp = pd.read_excel(f, header=0)
            df_temp.columns = df_temp.columns.astype(str).str.strip()
            df_temp['source_file'] = os.path.basename(f)
            dfs.append(df_temp)
            print(f" ✅ Loaded: {os.path.basename(f)}")
        except Exception as e:
            print(f" ❌ Error: {os.path.basename(f)} : {str(e)[:50]}")
    df = pd.concat(dfs, ignore_index=True)
    print(f"Total rows loaded: {len(df)}")
    return df

# ----------------------------
# 3️⃣ MAP QUESTION LABELS → STANDARD COLUMN NAMES
# ----------------------------
questions_mapping = {
    "triage_raw":              [r"Tri\s*IAO", r"Score\s*de\s*Gravit[ée]", r"Niveau\s*de\s*Triage"],
    "transport":               [r"Mode\s*d.arriv[ée]e", r"Mode\s*de\s*Transport"],
    "atcd_ioa":                [r"Ant[ée]c[ée]dents"],
    "chief_complaint":         [r"Motifs?\s*de\s*recours", r"Motif\s*de\s*consultation"],
    "anam_ioa":                [r"Circonstances", r"Commentaires\s*aux\s*urgences",
                                r"Histoire\s*de\s*la\s*maladie", r"Anamn[èe]se"],
    "ttt_adm_ioa_file":        [r"Traitement\s*administr[ée]", r"Traitement\s*en\s*cours"],
    "admission_summary_ioa":   [r"Synth[èe]se\s*PEC", r"Synth[èe]se\s*Initiale"],
    "evolution_ioa":           [r"Evolution"],
    "date_hour_triage_begin":  [r"Date\s*et\s*heure"],
}

def match_question(label):
    if pd.isna(label):
        return None
    s = str(label).strip()
    for std_name, patterns in questions_mapping.items():
        for pattern in patterns:
            if re.search(pattern, s, re.IGNORECASE):
                return std_name
    return None

# ----------------------------
# 4️⃣ IDENTIFY COLUMN NAMES
# ----------------------------
def find_col(df, patterns):
    for p in patterns:
        for col in df.columns:
            if re.search(p, str(col), re.IGNORECASE):
                return col
    return None

# ----------------------------
# 5️⃣ HARMONIZE TRIAGE SCORE
# ----------------------------
def harmonize_triage(x):
    if pd.isna(x): return np.nan
    s = unidecode.unidecode(str(x)).lower().strip()

    # --- LEVEL 1 ---
    if any(k in s for k in ["reanim", "sans delai", "medecin <1min", "medecin < 1min"]):
        return "1"

    # --- LEVEL 2 ---
    if any(k in s for k in ["tres urgent", "très urgent", "medecin < 20min", "medecin <20min",
                              "infirmiere <10min", "infirmière <10min"]):
        return "2"

    # --- LEVEL 4 (avant niveau 3 pour éviter que "urgent" attrape "peu urgent") ---
    if any(k in s for k in ["peu urgent", "medecin <2h", "medecin < 2h",
                              "medecin <120min", "medecin < 120min", "120min"]):
        return "4"

    # --- LEVEL 5 ---
    if any(k in s for k in ["non urgent", "medecin <3h", "medecin < 3h",
                              "medecin <240min", "medecin < 240min", "240min"]):
        return "5"

    # --- LEVEL 3 (en dernier car "urgent" est présent dans niveau 2 et 4) ---
    if any(k in s for k in ["urgent", "medecin <1h", "medecin < 1h",
                              "medecin <60min", "medecin < 60min",
                              "medecin <90min", "medecin < 90min",
                              "60min", "90min"]):
        return "3"

    return np.nan

# ----------------------------
# 6️⃣ MAIN SCRIPT
# ----------------------------
def main():
    files = get_ioa_files()
    df_raw = load_files(files)

    # --- Identify key columns ---
    col_nda        = find_col(df_raw, [r"NDA", r"N°\s*DA", r"N°\s*dossier"])
    col_question   = find_col(df_raw, [r"Libell[ée]\s*question"])
    col_value      = find_col(df_raw, [r"Libell[ée]\s*court\s*ou\s*long", r"Libell[ée]\s*long"])
    col_date_adm   = find_col(df_raw, [r"Date\s*Entr[ée]e\s*S[ée]jour\s*avec\s*heure", r"Date\s*Entr[ée]e\s*S[ée]jour"])
    col_sex        = find_col(df_raw, [r"Sexe"])
    col_age        = find_col(df_raw, [r"Age"])
    col_year       = find_col(df_raw, [r"Ann[ée]e\s*entr[ée]e"])
    col_uam        = find_col(df_raw, [r"Code\s*UAM"])
    col_date_triage_end = find_col(df_raw, [r"Date\s*cr[ée]ation\s*questionnaire",
                                            r"Date\s*creation\s*questionnaire",
                                            r"Date\s*cr\u00e9ation"   # ← unicode explicite pour é
                                            ])

    print(f"col_date_triage_end → {col_date_triage_end}")
    print(f"patient_cols sera → {[c for c in [col_nda, col_date_adm, col_sex, col_age, col_uam, col_date_triage_end, 'source_file'] if c]}")


    print("\n=== Column mapping detected ===")
    for name, col in [("nda", col_nda), ("question", col_question), ("value", col_value),
                      ("date_adm", col_date_adm), ("sex", col_sex), ("age", col_age), ("date_triage_end", col_date_triage_end)]:
        print(f"  {name:12} → {col}")

    # --- Map question labels ---
    df_raw["question_std"] = df_raw[col_question].apply(match_question)

    print("\n=== Question label mapping check ===")
    print(df_raw.groupby(col_question)["question_std"].first().to_string())

    # --- Verify 'Date et heure' is captured ---
    print("\n=== Verification: 'Date et heure' pattern capture ===")
    mask = df_raw[col_question].str.contains(r"Date\s*et\s*heure", case=False, na=False)
    print(df_raw[mask][col_question].value_counts().to_string())

    df_known = df_raw[df_raw["question_std"].notna()].copy()
    print(f"\nRows after question filtering: {len(df_known):,} / {len(df_raw):,}")

    # --- Pivot long → wide ---
    patient_cols = [c for c in [col_nda, col_date_adm, col_sex, col_age, col_year, col_uam, col_date_triage_end, "source_file"] if c]

    df_pivot = df_known.pivot_table(
        index=patient_cols,
        columns="question_std",
        values=col_value,
        aggfunc="first"
    ).reset_index()
    df_pivot.columns.name = None

    # --- Rename patient columns ---
    rename_map = {}
    if col_nda:      rename_map[col_nda]      = "nda"
    if col_date_adm: rename_map[col_date_adm] = "date_adm_ioa_file"
    if col_sex:      rename_map[col_sex]       = "sex_ioa"
    if col_age:      rename_map[col_age]       = "age_ioa"
    if col_date_triage_end: rename_map[col_date_triage_end] = "date_hour_triage_end"
    df_pivot.rename(columns=rename_map, inplace=True)

    # --- Cleaning ---
    df_pivot["date_adm_ioa_file"] = pd.to_datetime(df_pivot["date_adm_ioa_file"], errors="coerce")
    df_pivot["age_ioa"]      = df_pivot["age_ioa"].astype(str).str.extract(r'(\d+)').astype(float)
    df_pivot["nda"]          = df_pivot["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

    if "date_hour_triage_begin" in df_pivot.columns:
        df_pivot["date_hour_triage_begin"] = pd.to_datetime(df_pivot["date_hour_triage_begin"], errors="coerce")

    # --- Triage duration ---
    if "date_hour_triage_begin" in df_pivot.columns and "date_hour_triage_end" in df_pivot.columns:
        df_pivot["date_hour_triage_end"] = pd.to_datetime(df_pivot["date_hour_triage_end"], errors="coerce")
        df_pivot["duration_triage_ioa_min"] = (
            df_pivot["date_hour_triage_end"] - df_pivot["date_hour_triage_begin"]
        ).dt.total_seconds() / 60
        print(f"\n=== Triage duration (min) ===")
        print(df_pivot["duration_triage_ioa_min"].describe().round(1))

    # --- Filter 2021–2025 ---
    df_pivot = df_pivot[
        (df_pivot["date_adm_ioa_file"].dt.year >= YEAR_MIN) &
        (df_pivot["date_adm_ioa_file"].dt.year <= YEAR_MAX)
    ].copy()
    print(f"\nRows after year filter ({YEAR_MIN}-{YEAR_MAX}): {len(df_pivot):,}")

    # --- Hospital ---
    df_pivot["hospital"] = np.where(
        df_pivot["source_file"].str.contains("SA", case=False, na=False), "SA", "PEL"
    )

    # --- Year column ---
    df_pivot["year"] = df_pivot["date_adm_ioa_file"].dt.year

    # --- Triage harmonization ---
    if "triage_raw" in df_pivot.columns:
        df_pivot["triage"] = df_pivot["triage_raw"].apply(harmonize_triage)

    # --- Chief complaint cleaning ---
    if "chief_complaint" in df_pivot.columns:
        df_pivot["chief_complaint"] = df_pivot["chief_complaint"].astype(str).str.replace(r".*-\s*", "", regex=True)
        df_pivot["chief_complaint"] = df_pivot["chief_complaint"].replace(['nan', 'None', ''], 'Unknown')

    # --- Final columns ---
    final_cols = [c for c in [
        "nda", "age_ioa", "sex_ioa", "hospital", "year", "date_adm_ioa_file",
        "date_hour_triage_begin", "date_hour_triage_end", "duration_triage_ioa_min",
        "triage", "triage_raw", "transport", "chief_complaint",
        "anam_ioa", "atcd_ioa", "ttt_adm_ioa_file",
        "admission_summary_ioa", "evolution_ioa"
    ] if c in df_pivot.columns]

    df_final = df_pivot[final_cols].sort_values("date_adm_ioa_file").drop_duplicates(subset="nda")

    # --- Export ---

    df_final.to_csv(output_file, index=False)

    print(f"\n=== Triage score distribution ===")
    print(df_final["triage"].value_counts(dropna=False).sort_index())

    print(f"\n=== Patients per hospital and year ===")
    print(df_final.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

    print(f"\n🚀 Done! {len(df_final):,} unique patients ({YEAR_MIN}-{YEAR_MAX}).")
    print(f"📁 Saved to: {output_file}")
    print(f"Columns: {df_final.columns.tolist()}")

    return df_final

# ===========================
if __name__ == "__main__":
    df_final = main()

# ============================================
# INVESTIGATION 1: True NaN (triage AND triage_raw = NaN)
# ============================================
true_nan = df_final[df_final["triage"].isna() & df_final["triage_raw"].isna()]

print(f"\n{'='*55}")
print(f"  INVESTIGATION 1: True NaN (triage AND triage_raw = NaN)")
print(f"{'='*55}")
print(f"Total patients             : {len(df_final):,}")
print(f"True NaN (no triage at all): {len(true_nan):,} ({len(true_nan)/len(df_final)*100:.1f}%)")

print("\n--- True NaN per hospital ---")
print(true_nan["hospital"].value_counts(dropna=False).to_string())

print("\n--- True NaN per year ---")
print(true_nan["year"].value_counts(dropna=False).sort_index().to_string())

print("\n--- True NaN per hospital AND year ---")
print(true_nan.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

# ============================================
# INVESTIGATION 2: NaN in date_hour_triage_begin
# ============================================
print(f"\n{'='*55}")
print(f"  INVESTIGATION 2: NaN in date_hour_triage_begin")
print(f"{'='*55}")

if "date_hour_triage_begin" in df_final.columns:
    nan_triage_begin = df_final[df_final["date_hour_triage_begin"].isna()]

    print(f"Total patients                   : {len(df_final):,}")
    print(f"NaN date_hour_triage_begin        : {len(nan_triage_begin):,} ({len(nan_triage_begin)/len(df_final)*100:.1f}%)")

    print("\n--- NaN date_hour_triage_begin per hospital ---")
    print(nan_triage_begin["hospital"].value_counts(dropna=False).to_string())

    print("\n--- NaN date_hour_triage_begin per year ---")
    print(nan_triage_begin["year"].value_counts(dropna=False).sort_index().to_string())

    print("\n--- NaN date_hour_triage_begin per hospital AND year ---")
    print(nan_triage_begin.groupby(["hospital", "year"]).size().unstack(fill_value=0).to_string())

    print("\n--- Fill rate (%) of date_hour_triage_begin per hospital AND year ---")
    total  = df_final.groupby(["hospital", "year"]).size().unstack(fill_value=0)
    manque = nan_triage_begin.groupby(["hospital", "year"]).size().unstack(fill_value=0)
    taux   = ((1 - manque / total) * 100).round(1)
    print(taux.to_string())
else:
    print("⚠️  Column 'date_hour_triage_begin' not found — check the exact label in source files.")

--- INVENTORY: 2 files detected ---


 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2022.xlsx


 ✅ Loaded: CGJ 030a - Dossier IOA complet - 2023.xlsx
Total rows loaded: 1420242
col_date_triage_end → Date création questionnaire
patient_cols sera → ['NDA', 'Date Entrée Séjour avec heure', 'Sexe patient', "Age à l'entrée du séjour", 'Code UAM entrée séjour', 'Date création questionnaire', 'source_file']

=== Column mapping detected ===
  nda          → NDA
  question     → Libellé question
  value        → Libellé court ou long
  date_adm     → Date Entrée Séjour avec heure
  sex          → Sexe patient
  age          → Age à l'entrée du séjour
  date_triage_end → Date création questionnaire



=== Question label mapping check ===
Libellé question
A jeun                                                                         None
Aide(s) à domicile                                                             None
Allergies                                                                      None
Antécédents                                                                atcd_ioa
Autonomie                                                                      None
Autorisation parentale signée                                                  None
Bon de transport                                                               None
Circonstances                                                              anam_ioa
Compte-rendu d'imagerie et CD                                                  None
Courrier médical                                                               None
Courrier paramédical                                                           None
Critère(s) interventi

Libellé question
Date et heure    81653



Rows after question filtering: 547,090 / 1,420,242



=== Triage duration (min) ===
count     32528.0
mean      -9776.6
std      205106.3
min     -465120.0
25%     -167038.0
50%           3.0
75%      128163.0
max      465153.0
Name: duration_triage_ioa_min, dtype: float64

Rows after year filter (2022-2025): 81,656



=== Triage score distribution ===
triage
1        321
2      18713
3      29209
4      26653
5       6493
NaN        7
Name: count, dtype: int64

=== Patients per hospital and year ===
year       2022   2023
hospital              
PEL       43431  37965

🚀 Done! 81,396 unique patients (2022-2025).
📁 Saved to: Datasets/df_ioa_2022_2025.csv
Columns: ['nda', 'age_ioa', 'sex_ioa', 'hospital', 'year', 'date_adm_ioa_file', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'triage', 'triage_raw', 'transport', 'chief_complaint', 'anam_ioa', 'atcd_ioa', 'ttt_adm_ioa_file', 'admission_summary_ioa', 'evolution_ioa']

  INVESTIGATION 1: True NaN (triage AND triage_raw = NaN)
Total patients             : 81,396
True NaN (no triage at all): 7 (0.0%)

--- True NaN per hospital ---
hospital
PEL    7

--- True NaN per year ---
year
2022    3
2023    4

--- True NaN per hospital AND year ---
year      2022  2023
hospital            
PEL          3     4

  INVESTIGATION 2: Na

In [8]:
# Voir les triage_raw non mappés
print("=== triage_raw values NOT mapped (NaN after harmonize) ===")
mask_nan = df_final["triage"].isna() & df_final["triage_raw"].notna()
print(df_final[mask_nan]["triage_raw"].value_counts().to_string())

print("\n=== ALL triage_raw values ===")
print(df_final["triage_raw"].value_counts(dropna=False).to_string())

=== triage_raw values NOT mapped (NaN after harmonize) ===
Series([], )

=== ALL triage_raw values ===
triage_raw
Urgent (médecin <1h)            29209
Peu urgent (médecin <2h)        26653
Très urgent (médecin <20min)    18713
Non urgent (médecin <3h)         6493
Réanimation (médecin <1min)       321
NaN                                 7


In [9]:
# ============================================
# INVESTIGATION 3: triage_raw values that failed mapping (triage = NaN but triage_raw is not NaN)
# ============================================
print(f"\n{'='*55}")
print(f"  INVESTIGATION 3: triage_raw values not mapped")
print(f"{'='*55}")

if "triage_raw" in df_final.columns:
    unmapped = df_final[df_final["triage"].isna() & df_final["triage_raw"].notna()]
    print(f"Patients with triage_raw filled but triage = NaN: {len(unmapped):,}")

    print("\n--- Unmapped triage_raw values (sorted by frequency) ---")
    print(unmapped["triage_raw"].value_counts(dropna=False).to_string())


  INVESTIGATION 3: triage_raw values not mapped
Patients with triage_raw filled but triage = NaN: 0

--- Unmapped triage_raw values (sorted by frequency) ---
Series([], )


In [10]:
print(df_final['triage'].value_counts(dropna=False).to_string())

print("=== Triage distribution par année ===")
print(df_final.groupby(["year", "triage"])["nda"].count().unstack(fill_value=0).to_string())

print("\n=== Triage distribution par année (%) ===")
triage_pct = (
    df_final.groupby(["year", "triage"])["nda"]
    .count()
    .unstack(fill_value=0)
    .apply(lambda row: (row / row.sum() * 100).round(1), axis=1)
)
print(triage_pct.to_string())

triage
3      29209
4      26653
2      18713
5       6493
1        321
NaN        7
=== Triage distribution par année ===
triage    1     2      3      4     5
year                                 
2022    176  9108  15713  14839  3592
2023    145  9605  13496  11814  2901

=== Triage distribution par année (%) ===
triage    1     2     3     4    5
year                              
2022    0.4  21.0  36.2  34.2  8.3
2023    0.4  25.3  35.6  31.1  7.6


In [11]:
print("\n--- All triage_raw unique values ---")
print(df_final["triage_raw"].value_counts(dropna=False).to_string())


--- All triage_raw unique values ---
triage_raw
Urgent (médecin <1h)            29209
Peu urgent (médecin <2h)        26653
Très urgent (médecin <20min)    18713
Non urgent (médecin <3h)         6493
Réanimation (médecin <1min)       321
NaN                                 7



# 3. Medical files #


## New medical files, from 2022 to 2023, long format excel ##

In [18]:
import os
import glob
import pandas as pd
import numpy as np
import re



#------CONSTANTS--------
DATA_PATH = "../Datanad/subset_data"
YEAR_MIN = 2022
YEAR_MAX = 2023
os.makedirs("Datasets", exist_ok=True)
output_file = f"Datasets/df_med_{YEAR_MIN}_{YEAR_MAX}.csv"


# --- Step 1 : Load and read ---
pattern = os.path.join(DATA_PATH, "CGJ_055*")
all_files = glob.glob(pattern)
files = [f for f in all_files if not os.path.basename(f).startswith("~$")]

if not files:
    print("No file found.")
    exit()

dfs = []
for file in files:
    try:
        df = pd.read_excel(file)
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            cols[cols == dup] = [f"{dup}_{i}" if i != 0 else dup for i in range(len(cols[cols == dup]))]
        df.columns = cols
        df['source_file'] = os.path.basename(file)
        dfs.append(df)
        print(f"✅ Loaded : {os.path.basename(file)} ({len(df)} lines)")
    except Exception as e:
        print(f"❌ Error on {os.path.basename(file)} : {type(e).__name__} — {e}")

print(f"\ndfs contient {len(dfs)} dataframe(s)")
print(f"Fichiers tentés : {[os.path.basename(f) for f in files]}")

df_concat = pd.concat(dfs, ignore_index=True)
print(f"\nConcatenation done. Total : {len(df_concat)} lines")

# --- Step 2 : Normalize column names (strip spaces/accents issues) ---
df_concat.columns = df_concat.columns.str.strip()

# Identify the key columns by flexible matching
def find_col(df, pattern):
    """Return first column name matching a regex pattern (case-insensitive)."""
    for col in df.columns:
        if re.search(pattern, str(col), re.IGNORECASE):
            return col
    return None

col_nda        = find_col(df_concat, r"nda")
col_sex        = find_col(df_concat, r"sexe")
col_age = find_col(df_concat, r"age.*entr.e")
col_date_adm = find_col(df_concat, r"date\s+entr.e\s+s.jour\s+avec") # with time
col_date_creation = find_col(df_concat, r"date\s+cr.ation")
col_diag       = find_col(df_concat, r"diagnostic")
col_question   = find_col(df_concat, r"libell.\s+question")                # question label  → future column names
col_value      = find_col(df_concat, r"libell.\s+(court|long)")            # answer content  → future values
col_uam        = find_col(df_concat, r"uam")


print("\n--- Detected columns ---")
for name, val in [("NDA", col_nda), ("Sex", col_sex), ("Admission date", col_date_adm), ("Medical visit date", col_date_creation),
                  ("Diagnostic", col_diag), ("Question label", col_question),
                  ("Answer value", col_value)]:
    print(f"  {name:15s} → {val}")

# --- Step 3 : Pivot long → wide ---
# The pivot key is NDA (one patient = one row after pivot)
# Each unique value of col_question becomes a column, filled with col_value

# Keep metadata columns that are constant per NDA (one value per patient)
meta_cols = [c for c in [col_nda, col_sex, col_age, col_date_adm, col_date_creation, col_diag, col_uam, "source_file"]
             if c is not None]

# Deduplicate metadata (keep first occurrence per NDA)
df_meta = df_concat[meta_cols].drop_duplicates(subset=[col_nda])

# Pivot the question/answer pairs
if col_question and col_value and col_nda:
    df_pivot = df_concat[[col_nda, col_question, col_value]].copy()
    df_pivot = df_pivot.dropna(subset=[col_question])

    # Normalize question labels for clean column names
    df_pivot[col_question] = (df_pivot[col_question]
                               .astype(str)
                               .str.strip()
                               .str.lower()
                               .str.replace(r'\s+', '_', regex=True)
                               .str.replace(r'[^a-z0-9_àâäéèêëîïôùûü]', '', regex=True))

    # Keep last non-null value if a patient has multiple entries for the same question
    df_pivot = df_pivot.drop_duplicates(subset=[col_nda, col_question], keep='last')

    df_wide = df_pivot.pivot(index=col_nda, columns=col_question, values=col_value).reset_index()
    df_wide.columns.name = None
else:
    print("⚠️  Could not find question/value/NDA columns — pivot skipped.")
    df_wide = pd.DataFrame({col_nda: df_concat[col_nda].unique()})

# --- Step 4 : Merge metadata + pivoted questions ---
df_merged = df_meta.merge(df_wide, on=col_nda, how='left')

# --- Step 5 : Rename to standard column names ---
# Map detected metadata columns
rename_map = {}
if col_nda:        rename_map[col_nda]        = "nda"
if col_sex:        rename_map[col_sex]        = "sex"
if col_age:        rename_map[col_age] = "age"
if col_date_adm: rename_map[col_date_adm] = "date_adm_med"
if col_date_creation:  rename_map[col_date_creation]  = "date_hour_medical_visit"
if col_diag:       rename_map[col_diag]       = "diag"


df_merged.rename(columns=rename_map, inplace=True)

# --- Step 6 : Map pivoted question columns → standard names ---
# These regexes match the normalized question labels created during pivot
question_col_mapping = {
    "anam_ed":        [r"histoire", r"anamn.se"],
    "atcd_med":       [r"ant.c.dent", r"atcd"],
    "rx_home":        [r"traitement.*(habituel|entr.e|domicile)"],
    "clinical_exam":  [r"examen.clinique"],
    "rx_ed":          [r"traitement.*administr", r"actes.*th.rapeutiques", r"soins.*urgence"],
    "evolution":      [r".volution"],
    "conclusion":     [r"conclusion"],
    "ccmu":           [r"ccmu"],
    "disposition_med":    [r"devenir"],
    "additional_tests": [r"examens?.compl.mentaires?"]
}

for standard_name, patterns in question_col_mapping.items():
    if standard_name in df_merged.columns:
        continue  # already present from metadata
    for pat in patterns:
        matched = [c for c in df_merged.columns if re.search(pat, str(c), re.IGNORECASE)]
        if matched:
            # Merge all matches into one column (combine_first for non-null priority)
            series = df_merged[matched[0]]
            for extra in matched[1:]:
                series = series.combine_first(df_merged[extra])
            df_merged[standard_name] = series
            break  # stop at first pattern group that matched

# --- Step 7 : Final column selection ---
# We make sure "age" is in the list
final_cols = ["nda", "sex", "age", "date_adm_med", "date_hour_medical_visit",
              "diag", "anam_ed", "atcd_med", "rx_home", "clinical_exam",
              "rx_ed", "evolution", "additional_tests", "conclusion",
              "ccmu", "disposition_med", "source_file"]

df_final = df_merged[[c for c in final_cols if c in df_merged.columns]].copy()

# Clean Age (ensure it's a number)
if "age" in df_final.columns:
    df_final["age"] = pd.to_numeric(df_final["age"], errors='coerce')

# Clean NDA
df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', np.nan)

# --- Step 8 : Export & Saving ---
df_final.to_csv(output_file, index=False)
print(f"\n✅ Saved global file to: {output_file}")

# # --- Step 9 : Subset 2022 for the root environment ---
# print("📦 Creating 2022 medical subset...")
# date_col = "date_adm_med" if "date_adm_med" in df_final.columns else "date_hour_medical_visit"
#
# if date_col in df_final.columns:
#     temp_dates = pd.to_datetime(df_final[date_col], errors='coerce')
#     df_med_2022 = df_final[temp_dates.dt.year == 2022].copy()
#
#     subset_med_csv = "df_med_subset22pel.csv"
#     df_med_2022.to_csv(subset_med_csv, index=False)
#     print(f"✅ Subset 2022 created with {len(df_med_2022)} rows (including age).")

✅ Loaded : CGJ_055b_-_Dossier_patient_full_2023.xlsx (535881 lines)


✅ Loaded : CGJ_055b_-_Dossier_patient_full_2022.xlsx (574886 lines)

dfs contient 2 dataframe(s)
Fichiers tentés : ['CGJ_055b_-_Dossier_patient_full_2023.xlsx', 'CGJ_055b_-_Dossier_patient_full_2022.xlsx']

Concatenation done. Total : 1110767 lines

--- Detected columns ---
  NDA             → NDA
  Sex             → Sexe patient
  Admission date  → Date Entrée Séjour avec heure
  Medical visit date → Date création questionnaire
  Diagnostic      → Code et libellé diagnostic
  Question label  → Libellé question
  Answer value    → Libellé court ou long



✅ Saved global file to: Datasets/df_med_2022_2025.csv


In [19]:
# --- Unique rows check ---
print(f"\n=== UNIQUE ROWS CHECK ===")
print(f"Total rows          : {len(df_final):,}")
print(f"Unique NDA          : {df_final['nda'].nunique():,}")
print(f"Duplicate NDA       : {len(df_final) - df_final['nda'].nunique():,}")


=== UNIQUE ROWS CHECK ===
Total rows          : 60,289
Unique NDA          : 60,289
Duplicate NDA       : 0



# 4. Radio #


## 2022 to 2023 ##

In [21]:
# ===================================================
# Filtering and pivoting radio exam files 2022-2025
# ===================================================
import pandas as pd
import glob
import os

# --- Paths ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2023
output_csv = os.path.join(OUTPUT_PATH, f"df_radio_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Step 1: Load all CGJ 073* files ---
all_files = glob.glob(os.path.join(DATA_PATH, "*"))
files = [f for f in all_files
         if os.path.basename(f).startswith("CGJ 073")
         and not os.path.basename(f).startswith("~$")]

print(f"{len(files)} file(s) found:")
for f in files:
    print(f"  - {os.path.basename(f)}")

# --- Step 2: Load, filter by year range and concatenate ---
dfs = []
for file in files:
    name = os.path.basename(file)
    try:
        df_raw = pd.read_excel(file, header=3)
        df_raw.columns = df_raw.columns.astype(str).str.strip()

        date_col = "Date Entree"
        if date_col not in df_raw.columns:
            print(f"⚠️  '{date_col}' not found in {name} — columns: {df_raw.columns.tolist()}")
            continue

        df_raw[date_col] = pd.to_datetime(df_raw[date_col], dayfirst=True, errors='coerce')
        df_filtered = df_raw[
            (df_raw[date_col].dt.year >= YEAR_START) &
            (df_raw[date_col].dt.year <= YEAR_END)
        ].copy()

        df_filtered["source_file"] = name
        dfs.append(df_filtered)
        print(f"✅ {name} — {len(df_filtered)} rows after {YEAR_START}-{YEAR_END} filter")

    except Exception as e:
        print(f"❌ {name} — {type(e).__name__}: {e}")

if not dfs:
    raise ValueError("No files loaded — check paths and column names.")

df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.astype(str).str.strip()
print(f"\nTotal after concat: {len(df)} rows")

# --- Step 3: Deduplicate exams ---
date_col       = "Date Entree"
exam_label_col = "Libellé examen"
exam_type_col  = "Libellé Type examen"
report_col     = "Compte rendu (cr) pour recherche texte (4000 caractères)"
patient_col    = "Numéro de venue (Xplore)"

# Check all key columns exist
for col in [date_col, exam_label_col, exam_type_col, report_col, patient_col]:
    if col not in df.columns:
        print(f"⚠️  Missing column: '{col}'")

key_cols = [date_col, exam_label_col, exam_type_col]

def keep_report_if_exists(group):
    """For each exam group, keep rows with a report if any exist, otherwise keep first row."""
    if report_col in group.columns:
        has_report = group[report_col].notna() & (group[report_col].astype(str).str.strip() != "")
        if has_report.any():
            return group[has_report]
    return group.iloc[[0]]

df_clean = (df.groupby(key_cols, as_index=False, group_keys=False)
              .apply(keep_report_if_exists)
              .reset_index(drop=True))

# --- Step 4: Rename key columns ---
df_clean.rename(columns={
    patient_col: "nda",
    date_col:    "admission_date"
}, inplace=True)

# --- Step 5: Create indices for each exam type per patient ---
df_clean["exam_index"]     = df_clean.groupby(["nda", exam_type_col]).cumcount() + 1
df_clean["date_col_new"]   = df_clean[exam_type_col] + "_date"   + df_clean["exam_index"].astype(str)
df_clean["exam_col_new"]   = df_clean[exam_type_col] + "_exam"   + df_clean["exam_index"].astype(str)
df_clean["report_col_new"] = df_clean[exam_type_col] + "_report" + df_clean["exam_index"].astype(str)

# --- Step 6: Pivot to wide format ---
df_dates   = df_clean.pivot(index=["nda", "admission_date"], columns="date_col_new",   values="Date heure examen")
df_exams   = df_clean.pivot(index=["nda", "admission_date"], columns="exam_col_new",   values=exam_label_col)
df_reports = df_clean.pivot(index=["nda", "admission_date"], columns="report_col_new", values=report_col)

df_wide = df_dates.join([df_exams, df_reports]).reset_index()
df_wide.columns.name = None

# --- Step 7: Clean NDA ---
df_wide["nda"] = df_wide["nda"].astype(str).str.replace(r'\.0$', '', regex=True)

# --- Step 8: Save ---
df_wide.to_csv(output_csv, index=False)
print(f"\n✅ Done! {len(df_wide)} unique patients saved to {output_csv}")


##

1 file(s) found:
  - CGJ 073 - Radio - 2224.xlsx


✅ CGJ 073 - Radio - 2224.xlsx — 58778 rows after 2022-2023 filter

Total after concat: 58778 rows


/tmp/ipykernel_108169/3586864145.py:83: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.




✅ Done! 37498 unique patients saved to Datasets/df_radio_2022_2023.csv


In [23]:
df_wide

,nda,admission_date,Consultation Imagerie_date1,Echographie_date1,Echographie_date2,IRM_date1,IRM_date2,IRM_date3,IRM_date4,Médecine nucléaire_date1,PEDIA HORS IMAGERIE_date1,Radiologie Interventionnelle_date1,Radiologie Interventionnelle_date2,Radiologie conventionnelle_date1,Radiologie conventionnelle_date2,Radiologie conventionnelle_date3,Radiologie conventionnelle_date4,Radiologie conventionnelle_date5,Tep Scan_date1,Tomodensitométrie_date1,Tomodensitométrie_date2,Tomodensitométrie_date3,Tomodensitométrie_date4,URGENCE HORS IMAGERIE_date1,Consultation Imagerie_exam1,Echographie_exam1,Echographie_exam2,IRM_exam1,IRM_exam2,IRM_exam3,IRM_exam4,Médecine nucléaire_exam1,PEDIA HORS IMAGERIE_exam1,Radiologie Interventionnelle_exam1,Radiologie Interventionnelle_exam2,Radiologie conventionnelle_exam1,Radiologie conventionnelle_exam2,Radiologie conventionnelle_exam3,Radiologie conventionnelle_exam4,Radiologie conventionnelle_exam5,Tep Scan_exam1,Tomodensitométrie_exam1,Tomodensitométrie_exam2,Tomodensitométrie_exam3,Tomodensitométrie_exam4,URGENCE HORS IMAGERIE_exam1,Consultation Imagerie_report1,Echographie_report1,Echographie_report2,IRM_report1,IRM_report2,IRM_report3,IRM_report4,Médecine nucléaire_report1,PEDIA HORS IMAGERIE_report1,Radiologie Interventionnelle_report1,Radiologie Interventionnelle_report2,Radiologie conventionnelle_report1,Radiologie conventionnelle_report2,Radiologie conventionnelle_report3,Radiologie conventionnelle_report4,Radiologie conventionnelle_report5,Tep Scan_report1,Tomodensitométrie_report1,Tomodensitométrie_report2,Tomodensitométrie_report3,Tomodensitométrie_report4,URGENCE HORS IMAGERIE_report1
0,22020804251,2022-08-28 01:42:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2022-08-28 23:34:00,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TDM CRANE + COU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"\n\n\nLEBAS, Laurent\nNé(e) le : 12/02/1990 (3...",NaN,NaN,NaN,NaN
1,22030011617,2022-01-01 00:08:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2022-01-01 02:05:00,NaT,NaT,NaT,NaT,NaT,2022-01-01 03:37:00,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RADIO BILAN URGENCE,NaN,NaN,NaN,NaN,NaN,TDM CRANE/CEREBRAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"\n\n\nVENDEVILLE, Thomas\nNé(e) le : 05/01/198...",NaN,NaN,NaN,NaN,NaN,"\n\n\nVENDEVILLE, Thomas\nNé(e) le : 05/01/198...",NaN,NaN,NaN,NaN
2,22030011619,2022-01-01 00:17:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2022-01-01 01:26:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RADIO THORAX AU LIT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22030011634,2022-01-01 01:00:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2022-01-01 09:28:00,2022-01-01 09:28:00,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TDM ANGIOSCAN THORAX,TDM ANGIOSCAN THORAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"\n\n\nGILMOUR, Prianka\nNé(e) le : 14/02/1992 ...","\n\n\nGILMOUR, Prianka\nNé(e) le : 14/02/1992 ...",NaN,NaN,NaN
4,22030011640,2022-01-01 01:31:00,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2022-01-01 03:32:00,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TDM CRANE/MASSIF FACIAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"\n\n\nEL KATTAT, Ahmed\nNé(e) le : 08/11/1995 ...",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19695,22039002233,2022-04-20 22:00:00,NaT,NaT,NaT,2022-04-21 03:18:00,

# 5. Biology #

### 2022 to 2025 ###

In [24]:
import os
import glob
import pandas as pd
import numpy as np

# --- Config ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2023
output_csv = os.path.join(OUTPUT_PATH, f"df_bio_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# #######################################################
# A) LOAD ALL GLIMS FILES
# #######################################################
def append_all_glims():
    all_glims_files = glob.glob(os.path.join(DATA_PATH, "GLI*.xlsx"))
    glims_files = [f for f in all_glims_files if not os.path.basename(f).startswith("~$")]

    if not glims_files:
        print("[GLIMS] No files found.")
        return pd.DataFrame()

    print(f"[GLIMS] {len(glims_files)} file(s) found:")
    for f in glims_files:
        print(f"  - {os.path.basename(f)}")

    dfs = []
    for f in glims_files:
        try:
            df = pd.read_excel(f, sheet_name="Analyses", header=1, dtype={"N° venue": str})

            # Filter by year range
            date_col = "Date Prélèvement (en date)"
            if date_col in df.columns:
                df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
                df = df[
                    (df[date_col].dt.year >= YEAR_START) &
                    (df[date_col].dt.year <= YEAR_END)
                ].copy()

            # Raw value backup with dot decimal
            raw_with_dot = df["Résultat brute de l'analyse"].astype(str).str.replace(',', '.', regex=False)

            # Merge result sources
            df['mixed_result'] = df["Résultat de l'analyse"].astype(str).replace('nan', np.nan)
            df['mixed_result'] = df['mixed_result'].fillna(raw_with_dot)

            df["source_file"] = os.path.basename(f)
            dfs.append(df)
            print(f"✅ {os.path.basename(f)} — {len(df)} rows after {YEAR_START}-{YEAR_END} filter")

        except Exception as e:
            print(f"❌ {os.path.basename(f)} — {type(e).__name__}: {e}")

    if not dfs:
        return pd.DataFrame()

    df_glims_raw = pd.concat(dfs, ignore_index=True)
    print(f"\n[GLIMS raw] Total: {df_glims_raw.shape[0]} rows.")
    return df_glims_raw

# #######################################################
# B) PROCESS GLIMS (PIVOT)
# #######################################################
def process_glims_pivot(df_glims_raw):
    # 1 - Rename columns
    rename_map = {
        "N° venue": "nda",
        "Date Prélèvement (en date)": "sample_date",
        "Libellé analyse détaillée": "analysis",
        "mixed_result": "result",
    }
    df_glims_raw.rename(columns=rename_map, inplace=True, errors="ignore")

    # 2 - Clean IDs and dates
    if "nda" in df_glims_raw.columns:
        df_glims_raw["nda"] = (df_glims_raw["nda"].astype(str)
                                .str.replace(r'\.0$', '', regex=True)
                                .str.strip())
    if "sample_date" in df_glims_raw.columns:
        df_glims_raw["sample_date"] = pd.to_datetime(df_glims_raw["sample_date"], errors="coerce")

    # 3 - Normalize analysis labels
    df_glims_raw["analysis"] = (df_glims_raw["analysis"].astype(str)
                                 .str.replace(r'\xa0', ' ', regex=True)
                                 .str.strip())
    analysis_lower = df_glims_raw["analysis"].str.lower()

    # Force explicit names
    mask_culture = df_glims_raw["analysis"] == "Culture"
    df_glims_raw.loc[mask_culture, "analysis"] = "culture_global"

    mask_gds = (analysis_lower.str.contains("origine", na=False) &
                analysis_lower.str.contains("gds", na=False))
    df_glims_raw.loc[mask_gds, "analysis"] = "gds_origin_global"

    lcr_mapping = {
        "Aspect du LCR": "lcr_aspect_1",
        "Asp.LCR centrifugé": "lcr_aspect_2",
        "Aspect": "lcr_aspect_3"
    }
    df_glims_raw["analysis"] = df_glims_raw["analysis"].replace(lcr_mapping)

    # 4 - Clean result values (vectorized)
    def clean_values_vectorized(df):
        result   = df['result'].astype(str).str.strip()
        analysis = df['analysis'].astype(str)

        mask_culture = analysis == "culture_global"
        mask_lcr     = analysis.isin(["lcr_aspect_1", "lcr_aspect_2", "lcr_aspect_3"])
        mask_gds     = analysis == "gds_origin_global"
        mask_empty   = result.str.lower().isin(["", "nan", "none"])
        mask_nren    = result.str.contains(r'\{<NREN', na=False, regex=True)

        cleaned = result.copy()

        # Culture
        cleaned.loc[mask_culture & mask_empty]            = np.nan
        cleaned.loc[mask_culture & ~mask_empty & ~mask_nren] = "YES"
        cleaned.loc[mask_culture & mask_nren]             = "NREN"

        # Empty (non-culture)
        cleaned.loc[mask_empty & ~mask_culture] = np.nan

        # GDS
        gds_mask_clean = mask_gds & ~mask_nren
        if gds_mask_clean.any():
            cleaned.loc[gds_mask_clean] = (cleaned.loc[gds_mask_clean]
                                            .str.replace('{', '', regex=False)
                                            .str.replace('}', '', regex=False)
                                            .str.replace('<', '', regex=False)
                                            .str.replace('BC_', '', regex=False)
                                            .str.strip())
        cleaned.loc[mask_gds & mask_nren] = "NREN"

        # Numeric extraction
        mask_numeric = ~(mask_culture | mask_lcr | mask_gds | mask_empty | mask_nren)
        if mask_numeric.any():
            numeric_extracted = cleaned.loc[mask_numeric].str.extract(r'(\d+\.?\d*)', expand=False)
            cleaned.loc[mask_numeric] = numeric_extracted.fillna(cleaned.loc[mask_numeric])

        # Numeric NREN
        cleaned.loc[mask_nren & ~(mask_culture | mask_lcr | mask_gds)] = "NREN"

        return cleaned

    df_glims_raw["result"] = clean_values_vectorized(df_glims_raw)

    # 5 - Analysis mapping
    analysis_map = {
        "Créatinine sg": "creatinine", "Urée sg": "urea",
        "Potassium sg": "potassium", "Sodium sg": "sodium", "Calcium sg arsenazo": "calcium",
        "Troponine I HS": "troponine", "CKMB": "ckmb", "BNP": "bnp",
        "ASAT (TGO)": "asat", "ALAT (TGP)": "alat", "Bilirubine totale": "bili_total",
        "PAL sg": "alp", "Lipase sg": "lipase",
        "Leucocytes": "leucocytes", "PNeutro Va": "neutrophils", "Lympho Va": "lymphocytes",
        "Monocytes Va": "monocytes", "PBaso Va": "basophils", "PEosino Va": "eosinophils",
        "Hémoglobine": "hemoglobine", "PlaquettesEDTA": "platelets",
        "TP Taux Prothrombine": "pt", "TCA patient": "aptt", "FibrinogèneClauss": "fibrinogen",
        "Activité AXa (HNF)": "axa_hnf", "Activité AXa Xarelto": "axa_xarelto",
        "Activité AXa Eliquis": "axa_eliquis", "Activité AXa HBPM": "axa_hbpm",
        "Anti-IIa Pradaxa": "aiia_pradaxa", "Activité AXa Arixtra": "axa_arixtra",
        "Activité AXa Orgaran": "axa_orgaran",
        "CK sg": "ck", "D-Dimères dosage": "ddimer",
        "Lactate": "lactates_a", "Lactate plasma vein": "lactates_v",
        "Calcium ionisé": "calcium_ionized",
        "Procalcitonine ser": "pct", "CRP": "crp",
        "Fer sg": "iron", "Ferritine": "ferritin",
        "culture_global": "culture",
        "lcr_aspect_1": "lcr_aspect_1", "lcr_aspect_2": "lcr_aspect_2", "lcr_aspect_3": "lcr_aspect_3",
        "gds_origin_global": "gds_origin",
        "pH(t)": "gds_ph", "pO2(t)": "gds_po2", "pCO2(t)": "gds_pco2",
        "Bicarbonates calculé": "gds_hco3", "SO2": "gds_so2"
    }

    # 6 - Filter and pivot
    df_filtered = df_glims_raw[df_glims_raw["analysis"].isin(analysis_map.keys())].copy()

    df_pivot = df_filtered.pivot_table(
        index=["nda", "sample_date"],
        columns="analysis",
        values="result",
        aggfunc="first"
    ).reset_index()

    # 7 - Rename to English
    df_pivot.rename(columns=analysis_map, inplace=True)
    df_pivot.columns.name = None

    return df_pivot

# #######################################################
# C) MAIN
# #######################################################
def main():
    print(f"\n=== Loading GLIMS files ({YEAR_START}-{YEAR_END}) ===")
    df_glims_raw = append_all_glims()

    if df_glims_raw.empty:
        print("No data loaded — exiting.")
        return pd.DataFrame()

    print("\n=== Pivot processing ===")
    df_glims = process_glims_pivot(df_glims_raw)

    # Keep earliest sample per patient
    df_glims.sort_values(by=["nda", "sample_date"], inplace=True)
    df_final = df_glims.groupby("nda", as_index=False).first()

    print(f"\n✅ Done: {df_final.shape[0]} unique patients.")
    return df_final

df_final = main()

# --- Summary ---
print("\n--- FINAL COLUMNS ---")
print(df_final.columns.tolist())

filling_rate = (df_final.notna().mean() * 100).round(2)
df_summary = filling_rate.to_frame(name="Filling Rate (%)").sort_values("Filling Rate (%)", ascending=False)
print("\n--- FILLING RATE SUMMARY ---")
print(df_summary)

# --- Save ---
df_final.to_csv(output_csv, index=False)
print(f"\n✅ Saved to: {output_csv}")


=== Loading GLIMS files (2022-2025) ===
[GLIMS] 6 file(s) found:
  - GLI-003 Requête_ponctuelle_Résultats 2023 0709.xlsx
  - GLI-003 Requête_ponctuelle_Résultats 2023 0406.xlsx
  - GLI-003 Requête_ponctuelle_Résultats 2023 0103.xlsx
  - GLI-003 Requête_ponctuelle_Résultats 2022 0712.xlsx
  - GLI-003 Requête_ponctuelle_Résultats 2023 1012.xlsx
  - GLI-003 Requête_ponctuelle_Résultats 2022 0106.xlsx


✅ GLI-003 Requête_ponctuelle_Résultats 2023 0709.xlsx — 558301 rows after 2022-2025 filter


✅ GLI-003 Requête_ponctuelle_Résultats 2023 0406.xlsx — 586836 rows after 2022-2025 filter


✅ GLI-003 Requête_ponctuelle_Résultats 2023 0103.xlsx — 446622 rows after 2022-2025 filter


✅ GLI-003 Requête_ponctuelle_Résultats 2022 0712.xlsx — 887639 rows after 2022-2025 filter


✅ GLI-003 Requête_ponctuelle_Résultats 2023 1012.xlsx — 425688 rows after 2022-2025 filter


✅ GLI-003 Requête_ponctuelle_Résultats 2022 0106.xlsx — 953700 rows after 2022-2025 filter



[GLIMS raw] Total: 3858786 rows.

=== Pivot processing ===



✅ Done: 46129 unique patients.

--- FINAL COLUMNS ---
['nda', 'sample_date', 'alat', 'asat', 'axa_hnf', 'axa_arixtra', 'axa_eliquis', 'axa_hbpm', 'axa_orgaran', 'axa_xarelto', 'aiia_pradaxa', 'bnp', 'gds_hco3', 'bili_total', 'ck', 'ckmb', 'crp', 'calcium_ionized', 'calcium', 'creatinine', 'ddimer', 'iron', 'ferritin', 'fibrinogen', 'hemoglobine', 'lactates_a', 'lactates_v', 'leucocytes', 'lipase', 'lymphocytes', 'monocytes', 'alp', 'basophils', 'eosinophils', 'neutrophils', 'platelets', 'potassium', 'pct', 'gds_so2', 'sodium', 'aptt', 'pt', 'troponine', 'urea', 'culture', 'gds_origin', 'lcr_aspect_1', 'lcr_aspect_2', 'lcr_aspect_3', 'gds_pco2', 'gds_ph', 'gds_po2']

--- FILLING RATE SUMMARY ---
                 Filling Rate (%)
nda                        100.00
sample_date                100.00
hemoglobine                 96.20
potassium                   95.72
sodium                      95.72
creatinine                  95.59
urea                        95.58
leucocytes             


✅ Saved to: Datasets/df_bio_2022_2025.csv


Bon je ne sais pas encore ce que e fais des paitents aui ont plusieurs antiXa differnent, on verra plus tard

# 6. Administrative Data #

## a. 2022 to 2023 ##

In [31]:
# ================================================================
# CODE FOR CSV FILE
#=========================================================================

# --- Config ---
DATA_PATH = "../Datanad/subset_data"
OUTPUT_PATH = "Datasets"
YEAR_START = 2022
YEAR_END = 2023
output_csv = os.path.join(OUTPUT_PATH, f"df_admin_{YEAR_START}_{YEAR_END}.csv")

os.makedirs(OUTPUT_PATH, exist_ok=True)

# --- Load CSV ---
csv_file = os.path.join(DATA_PATH, "CGJ_084_-_IMS_202225.csv")
df_final = pd.read_csv(csv_file, sep=";", encoding="utf-8-sig")
print(f"✅ Loaded: {len(df_final)} rows")
print("Colonnes trouvées :", df_final.columns.tolist())

# --- Rename ---
df_final.rename(columns={
    "UG entrée séjour Code":            "uam_service",
    "Nda":                              "nda",
    "Date entrée UG entrée séjour":     "date_entree_urg",
    "Date sortie UG entrée séjour":     "date_sortie_urg",
    "Date Sortie Venue":                "date_sortie_chu",
    "Urgence Date Sortie Completee":    "date_sortie_urg_completee",
    "Nom du patient":                   "nom",
    "Prénom du patient":                "prenom",
    "Date de naissance du patient":     "date_naissance",
    "Libelle de la ville de naissance": "ville_naissance",
    "Pays Naissance":                   "pays_naissance",
    "Adresse du patient (rue)":         "adresse_rue",
    "Code postal de la ville":          "adresse_cp",
    "Code commune de la ville":         "adresse_insee",
    "Libelle de la ville":              "adresse_ville",
    "Sejour Mode Sortie Libelle":       "mode_sortie_chu",
    "Décision urgence Libellé":         "decision_urgence",
}, inplace=True, errors="ignore")

# --- Filter UAM 9780 ---
n_before = len(df_final)
df_final = df_final[df_final["uam_service"].astype(str).str.strip() == "9780"].copy()
print(f"UAM filter: kept {len(df_final)} rows (removed {n_before - len(df_final)}).")

# --- Cleaning ---
df_final["nda"] = df_final["nda"].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_final["date_entree_urg"] = pd.to_datetime(df_final["date_entree_urg"],
                                          format="%Y/%m/%d %H:%M:%S", errors="coerce")

n_before = len(df_final)
df_final = df_final[
    (df_final["date_entree_urg"].dt.year >= YEAR_START) &
    (df_final["date_entree_urg"].dt.year <= YEAR_END)
].copy()
print(f"Year filter: removed {n_before - len(df_final)} rows outside {YEAR_START}-{YEAR_END}.")

# --- Dédoublonnage ---
df_final["count_nonnull"] = df_final.notna().sum(axis=1)
df_final.sort_values(by="count_nonnull", ascending=False, inplace=True)
n_before = len(df_final)
df_final.drop_duplicates(subset="nda", keep="first", inplace=True)
print(f"{n_before - len(df_final)} duplicates removed.")

# --- Sélection colonnes finales ---
cols_to_keep = {
    "uam_service":      "uam_service",
    "nda":              "nda",
    "date_naissance":   "date_naissance",
    "date_entree_urg":      "date_entree_urg",
    "date_sortie_urg":      "date_sortie_urg",
    "date_sortie_urg_completee": "date_sortie_urg_completee",
    "decision_urgence": "decision_urgence",
    "date_sortie_chu":      "date_sortie_chu",
    "mode_sortie_chu":      "mode_sortie_chu",
}

existing_cols = [c for c in cols_to_keep.keys() if c in df_final.columns]
df_admin = df_final[existing_cols].copy()
df_admin.sort_values(by="date_entree_urg", ascending=True, inplace=True)

print(f"\n✅ Done: {len(df_admin)} unique patients, {len(df_admin.columns)} colonnes.")

# --- Export ---
df_admin.to_csv(output_csv, index=False)
print(f"✅ Saved to: {output_csv}")

/tmp/ipykernel_108169/501403274.py:16: DtypeWarning:

Columns (8,15) have mixed types. Specify dtype option on import or set low_memory=False.



✅ Loaded: 341561 rows
Colonnes trouvées : ['Nda', 'Date de naissance du patient', 'Date entrée UG entrée séjour', 'Date sortie UG entrée séjour', 'Nom du patient', 'Prénom du patient', 'Adresse du patient (rue)', 'Code postal de la ville', 'Code commune de la ville', 'Libelle de la ville', 'Libellé du Hameau/Lieu-dit', 'Identifiant de la categorie professionnelle', 'Libelle de la ville de naissance', 'Pays Naissance', 'UG entrée séjour Code', 'Mode sortie séjour Libellé', 'Décision urgence Libellé', 'Année', 'Nationalité (Patient)', 'Date Sortie Venue', 'Sejour Mode Sortie Libelle', 'Sejour Mode Sortie Pmsi', 'Urgence Date Sortie Completee']


UAM filter: kept 337265 rows (removed 4296).


Year filter: removed 172076 rows outside 2022-2023.
74664 duplicates removed.

✅ Done: 90525 unique patients, 9 colonnes.


✅ Saved to: Datasets/df_admin_2022_2023.csv


In [32]:
print("mode_sortie_chu" in df_admin.columns)
print(df_admin["mode_sortie_chu"].value_counts(dropna=False).head(10))

True
mode_sortie_chu
NaN                                     46329
A - Domicile                            37403
B1 - Transfert MCO                       1698
B2 - Transfert SSR                       1636
C0 - Décédé                              1589
B4 - Transfert PSY                       1226
C1 - Transfert provisoire (<48h) MCO      458
A2 - Fugue                                101
B3 - Transfert Long Séjour                 76
D2 - Mutation SSR                           6
Name: count, dtype: int64
